# SPN Classifier with StatsBomb Open Data

This notebook introduces the SPN workflow using public event and 360 freeze-frame data from [StatsBomb Open Data](https://github.com/statsbomb/open-data). We select the 2022 World Cup matches as samples.

Load the data:

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display
from requests_cache import install_cache
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning

helper_directory = next(
    directory for directory in (
        Path.cwd(), Path.cwd() / "tests", Path.cwd() / "public_release" / "tests"
    ) if (directory / "data_checks.py").is_file()
)
sys.path.insert(0, str(helper_directory))
from data_checks import validate_match_catalogue

# Keep fetched data in memory during this session.
install_cache(backend="memory", expire_after=600)
warnings.filterwarnings("ignore", category=NoAuthWarning)

COMPETITION_ID = 43
SEASON_ID = 106

View the selected competition. Match information is stored in `matches_table` for the following steps.

In [2]:
competitions_table = sb.competitions(creds={})
selected_competition = competitions_table.loc[
    competitions_table["competition_id"].eq(COMPETITION_ID)
    & competitions_table["season_id"].eq(SEASON_ID),
    ["competition_id", "season_id", "competition_name", "season_name", "competition_gender"],
]
if selected_competition.empty:
    raise ValueError("The selected competition and season are unavailable in StatsBomb Open Data.")
display(selected_competition.reset_index(drop=True))

matches_table = sb.matches(
    competition_id=COMPETITION_ID, season_id=SEASON_ID, creds={}
).rename(columns={"competition_stage": "stage"})
matches_table = matches_table.sort_values(
    ["match_date", "kick_off", "match_id"]
).reset_index(drop=True)

,competition_id,season_id,competition_name,season_name,competition_gender
0,43,106,FIFA World Cup,2022,male


Run the catalogue checks defined in `data_checks.py`. The summary covers match metadata and event/360 availability. Eligible matches are stored in `matches_for_processing`.

In [3]:
integrity_summary, match_validation, matches_for_processing = validate_match_catalogue(
    matches_table, competition_id=COMPETITION_ID, season_id=SEASON_ID
)
display(integrity_summary)
print(f"Matches available for data loading: {len(matches_for_processing)} / {len(matches_table)}")

,Check,Passed,Flagged
0,Required fields populated,64,0
1,Positive integer match IDs,64,0
2,Unique match IDs,64,0
3,Selected competition and season,64,0
4,Valid date and kickoff time,64,0
5,Valid and distinct team IDs,64,0
6,Event data available,64,0
7,360 data available,64,0


Matches available for data loading: 64 / 64


Process all matches in `matches_for_processing` into pressure-anchor and outcome tables. Results are kept in `sequences_table`, `labels_table`, and `processing_log`. Raw match objects are retained in `match_data_by_id` for feature construction.

In [ ]:
import json

release_root = helper_directory.parent
sys.path.insert(0, str(release_root))
from spn import data_processing
from spn._network import PressureParams

model_config = json.loads((release_root / "model" / "model_config.json").read_text())
pressure_params = PressureParams(**model_config["pressure"])
sequence_config = data_processing.SequenceConfig(**model_config["sequence"])
label_config = data_processing.LabelConfig(**model_config["labels"])

In [ ]:
match_data_by_id = {}
sequence_parts = []
label_parts = []
processing_records = []

for match_id in matches_for_processing["match_id"]:
    events = list(sb.events(match_id=int(match_id), fmt="dict", creds={}).values())
    three_sixty = sb.frames(match_id=int(match_id), fmt="dict", creds={})
    match = data_processing.load_match_data(events, three_sixty, match_id=str(match_id))
    match_sequences, match_labels = data_processing.build_sequences_from_match(
        match, config=sequence_config, params=pressure_params, label_config=label_config
    )

    match_data_by_id[match.match_id] = match
    sequence_parts.append(match_sequences)
    label_parts.append(match_labels)
    processing_records.append({
        "match_id": match.match_id,
        "events": len(match.events),
        "360_records": match.raw_frame_count,
        "usable_frames": len(match.frames),
        "sequences": match_sequences["seq_id"].nunique() if not match_sequences.empty else 0,
        "anchors": len(match_sequences),
        "labels": len(match_labels),
        "warnings": len(match.warnings),
    })

sequences_table = pd.concat(sequence_parts, ignore_index=True) if sequence_parts else pd.DataFrame()
labels_table = pd.concat(label_parts, ignore_index=True) if label_parts else pd.DataFrame()
processing_log = pd.DataFrame(processing_records)

table_summaries = []
for name, table in {
    "sequences_table": sequences_table,
    "labels_table": labels_table,
    "processing_log": processing_log,
}.items():
    table_summaries.append({
        "Table": name,
        "Rows": len(table),
        "Columns": len(table.columns),
        "Matches": table["match_id"].nunique() if "match_id" in table else 0,
        "Missing cells": int(table.isna().sum().sum()),
        "Memory (MB)": round(table.memory_usage(deep=True).sum() / 1024**2, 2),
    })
table_summary = pd.DataFrame(table_summaries)
display(table_summary)

Build an SPN for each pressure anchor and prepare the 26 model inputs in the order specified by `model_config`. `model_input_table` contains these features and four event identifiers; per-match construction counts are saved in `feature_log`.

The summary counts missing values in the 26 input features. Only resolved sequences with representable anchors are scored; missing feature values remain unchanged.

In [ ]:
from spn.feature_engineering import build_feature_table

if sequences_table.empty:
    raise ValueError("No pressure sequences are available for network construction.")

anchor_columns = ["match_id", "seq_id", "ev_pos", "anchor_event_id"]
model_feature_columns = model_config["features"]
model_input_parts = []
feature_records = []
for match_id, match_sequences in sequences_table.groupby("match_id", sort=False):
    match_labels = labels_table.loc[labels_table["match_id"].eq(match_id)]
    match_features = build_feature_table(
        match_data_by_id[match_id],
        match_sequences,
        match_labels,
        params=pressure_params,
    )
    model_input_parts.append(match_features[anchor_columns + model_feature_columns].copy())
    feature_records.append({
        "match_id": match_id,
        **match_features.attrs["s3_completeness_audit"],
    })
    del match_features

model_input_table = pd.concat(model_input_parts, ignore_index=True)
feature_log = pd.DataFrame(feature_records)
feature_summary = pd.DataFrame([{
    "Table": "model_input_table",
    "Rows": len(model_input_table),
    "Columns": len(model_input_table.columns),
    "Matches": model_input_table["match_id"].nunique(),
    "Sequences": len(model_input_table[["match_id", "seq_id"]].drop_duplicates()),
    "Features": len(model_feature_columns),
    "Missing feature values": int(model_input_table[model_feature_columns].isna().sum().sum()),
    "Memory (MB)": round(model_input_table.memory_usage(deep=True).sum() / 1024**2, 2),
}])
display(feature_summary)
print(
    f"Complete sequences retained: {feature_summary.iloc[0]['Sequences']:,} / "
    f"{feature_log['expected_sequences'].sum():,}; "
    f"anchors retained: {len(model_input_table):,} / {len(sequences_table):,}."
)

Apply SPN Classifier to `model_input_table` and save predictions in `model_output_table`.

In [ ]:
from spn.model import SPNModel

model = SPNModel(config_path=release_root / "model" / "model_config.json")
model_output_table = model.predict(model_input_table)

Evaluate resolved rule-derived labels, giving each sequence total weight one. Unweighted anchor accuracy is reported separately.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
)

label_columns = anchor_columns + ["outcome_tag", "training_weight", "value_model_eligible"]
evaluation_candidates = model_output_table.merge(
    labels_table[label_columns].rename(columns={"outcome_tag": "observed_outcome"}),
    on=anchor_columns,
    how="left",
    validate="one_to_one",
)
evaluation_table = evaluation_candidates.loc[
    evaluation_candidates["value_model_eligible"].eq(True)
    & evaluation_candidates["observed_outcome"].isin(model.class_names)
].copy()
if evaluation_table.empty:
    raise ValueError("No observed fail, neutral, or success outcomes are available for evaluation.")

y_true = evaluation_table["observed_outcome"]
y_pred = evaluation_table["predicted_class"]
weights = evaluation_table["training_weight"]
probabilities = evaluation_table[[f"p_{name}" for name in model.class_names]]
class_weights = evaluation_table.groupby("observed_outcome")["training_weight"].sum()
majority_class = class_weights.idxmax()

evaluation_metrics = {
    "accuracy": accuracy_score(y_true, y_pred, sample_weight=weights),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred, sample_weight=weights),
    "macro_f1": f1_score(
        y_true, y_pred, labels=model.class_names, average="macro",
        sample_weight=weights, zero_division=0,
    ),
    "log_loss": log_loss(
        y_true, probabilities, labels=model.class_names, sample_weight=weights,
    ),
    "majority_baseline_accuracy": class_weights.max() / weights.sum(),
    "unweighted_anchor_accuracy": accuracy_score(y_true, y_pred),
}
precision, recall, class_f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=model.class_names, sample_weight=weights, zero_division=0,
)
evaluation_by_class = pd.DataFrame({
    "Class": model.class_names,
    "Precision": precision,
    "Recall": recall,
    "F1": class_f1,
    "Sequence support": support,
})
evaluation_summary = pd.DataFrame([
    {"Metric": "Accuracy (sequence-weighted)", "Value": f"{evaluation_metrics['accuracy']:.2%}"},
    {"Metric": "Balanced accuracy", "Value": f"{evaluation_metrics['balanced_accuracy']:.2%}"},
    {"Metric": "Macro F1", "Value": f"{evaluation_metrics['macro_f1']:.4f}"},
    {"Metric": "Log loss", "Value": f"{evaluation_metrics['log_loss']:.4f}"},
    {"Metric": f"Majority baseline accuracy ({majority_class})", "Value": f"{evaluation_metrics['majority_baseline_accuracy']:.2%}"},
    {"Metric": "Accuracy (unweighted anchors)", "Value": f"{evaluation_metrics['unweighted_anchor_accuracy']:.2%}"},
])
class_summary_rows = []
for row in evaluation_by_class.itertuples(index=False, name=None):
    class_name, class_precision, class_recall, score, sequence_count = row
    class_summary_rows.extend([
        {"Metric": f"{class_name} precision", "Value": f"{class_precision:.2%}"},
        {"Metric": f"{class_name} recall", "Value": f"{class_recall:.2%}"},
        {"Metric": f"{class_name} F1", "Value": f"{score:.4f}"},
    ])
evaluation_summary = pd.concat(
    [evaluation_summary, pd.DataFrame(class_summary_rows)], ignore_index=True,
).set_index("Metric")
display(evaluation_summary)


Average the calibrated success-minus-failure probabilities at sequence onset for each pressing team.

In [ ]:
from spn.analysis import team_values

team_inputs = model_output_table.merge(
    labels_table[anchor_columns + ["press_team_id"]].rename(columns={"press_team_id": "pressing_team_id"}),
    on=anchor_columns, validate="one_to_one",
)
team_summary = team_values(team_inputs).sort_values("expected_press_value", ascending=False)
display(team_summary.head())


## Counterfactual positioning experiments

Compare each eligible recorded presser with feasible nearby positions in one loaded match. The 2022 World Cup final is used when available.

In [ ]:
from spn import score_positions

example_id = "3869685" if "3869685" in match_data_by_id else next(iter(match_data_by_id))
position_scores = score_positions(match_data_by_id[example_id])
display(position_scores)
